# Clustering Track — K-Means (Day 1 Scaffold)
## 23CSE301 Machine Learning Capstone · Review 2
### Track: Clustering (Unsupervised Learning)
---
**Problem Context:**  
Discover natural operational groupings and machine condition regimes in the **AI4I 2020 Predictive Maintenance Dataset** using sensor readings without relying on ground-truth failure labels during fitting.

**Day 1 Scope:**
- Environment setup, dependencies, and modular utility import
- Dataset loading and feature audit
- Feature selection for clustering (strict exclusion of identifiers, targets, and failure subtypes to prevent data leakage)
- Feature scaling pipeline (`StandardScaler` fitted on the unsupervised feature matrix)
- K-Means clustering implementation ($k=3$, `k-means++`, `random_state=42`)
- Cluster label generation and distribution verification
- Day 1 validation guard checks and cluster profiling

> **ACADEMIC INTEGRITY NOTE:** Code cells provide scaffolding assistance. All analytical observations, interpretations of physical machine conditions, and conclusions in Markdown cells must be written by the student from inspection of actual outputs.

## 1. Environment Setup & Imports

In [1]:
# Cell 1.1 — Import core libraries and clustering utilities
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys
import os

warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)

# Ensure local directory and canonical src/clustering/ are in sys.path
CWD = os.getcwd()
candidate_src_paths = [
    CWD,
    os.path.abspath(os.path.join(CWD, '..', '..', 'src', 'clustering')),
    os.path.abspath(os.path.join(CWD, 'src', 'clustering')),
    os.path.abspath(os.path.join(CWD, '..', 'src', 'clustering')),
]
for p in candidate_src_paths:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

from clustering_utils import (
    validate_clustering,
    profile_clusters,
    compute_clustering_metrics,
)

# Reproducibility seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries and clustering utilities loaded successfully.")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__}")

Libraries and clustering utilities loaded successfully.
NumPy: 2.5.3 | Pandas: 3.0.5


## 2. Dataset Loading & Audit

In [2]:
# Cell 2.1 — Load dataset with robust path resolution
CANDIDATE_PATHS = [
    os.path.join('..', '..', 'data', 'ai4i2020.csv'),  # standard: Review2/notebooks/clustering/
    os.path.join('..', 'data', 'ai4i2020.csv'),        # from Review2/notebooks/
    os.path.join('data', 'ai4i2020.csv'),             # from Review2/
    'ai4i2020.csv',                                   # current dir
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError("Dataset ai4i2020.csv not found in candidate paths.")

df_raw = pd.read_csv(DATA_PATH)
print(f"Dataset loaded successfully from: {DATA_PATH}")
print(f"Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
df_raw.head(3)

Dataset loaded successfully from: ..\..\data\ai4i2020.csv
Shape: 10000 rows x 14 columns


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0


In [3]:
# Cell 2.2 — Sanity audit: data types and missing values
print("=== Column Types ===")
print(df_raw.dtypes)
print()
print("=== Missing Values ===")
print(df_raw.isnull().sum())
assert df_raw.isnull().sum().sum() == 0, "Unexpected missing values in dataset."

=== Column Types ===
UDI                          int64
Product ID                     str
Type                           str
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Machine failure              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
dtype: object

=== Missing Values ===
UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64


## 3. Feature Preparation for Clustering (Data Leakage Prevention)

**Crucial Architectural Decision:** Clustering must discover natural groupings purely from operational sensor measurements and machine characteristics.  

The following columns are **STRICTLY EXCLUDED** from the clustering feature matrix:

| Column | Reason for Exclusion |
|---|---|
| `UDI` | Arbitrary row index — not a physical feature |
| `Product ID` | Identifier string — not an operational measurement |
| `Machine failure` | Ground-truth binary target label — **using this in unsupervised fitting is a critical data-leakage violation** |
| `TWF`, `HDF`, `PWF`, `OSF`, `RNF` | Failure failure-mode breakdown labels — derived directly from failure status |

*Note:* `Type` is encoded ordinally ($L=0, M=1, H=2$) to reflect product quality variant variants.

In [4]:
# Cell 3.1 — Encode Type and construct feature matrix
df = df_raw.copy()
type_map = {'L': 0, 'M': 1, 'H': 2}
df['Type_encoded'] = df['Type'].map(type_map)

# Operational sensor features + product quality variant
CLUSTER_FEATURES = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
    'Type_encoded',
]

# Ground-truth labels stored separately for post-hoc validation ONLY (never passed to fit)
TRUE_LABELS = df['Machine failure'].values

X_cluster = df[CLUSTER_FEATURES].copy()
print(f"Clustering feature matrix shape: {X_cluster.shape}")
print(f"Selected features: {CLUSTER_FEATURES}")

Clustering feature matrix shape: (10000, 6)
Selected features: ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Type_encoded']


## 4. Feature Scaling

K-Means computes Euclidean distances between sample vectors and cluster centroids. Features with larger physical numeric ranges (such as Rotational Speed ~ 1500–2800 rpm vs Tool Wear ~ 0–250 min vs Type ~ 0–2) would dominate the distance calculation if unscaled.  
`StandardScaler` normalises each feature to zero mean and unit variance ($z = \frac{x - \mu}{\sigma}$).  
Because clustering is unsupervised, the scaler is fitted on the full clustering feature matrix.

In [5]:
# Cell 4.1 — Standardise features
cluster_scaler = StandardScaler()
X_scaled = cluster_scaler.fit_transform(X_cluster)

print("StandardScaler applied successfully.")
print(f"Scaled matrix shape: {X_scaled.shape}")
print(f"Feature means (target ~ 0): {X_scaled.mean(axis=0).round(4)}")
print(f"Feature stds  (target ~ 1): {X_scaled.std(axis=0).round(4)}")

StandardScaler applied successfully.
Scaled matrix shape: (10000, 6)
Feature means (target ~ 0): [-0.  0. -0.  0.  0. -0.]
Feature stds  (target ~ 1): [1. 1. 1. 1. 1. 1.]


## 5. K-Means Implementation

**Selecting k:**  
In Day 1, we implement an exploratory baseline with $k=3$ (corresponding naturally to the 3 machine product variants $L, M, H$).  
In Day 2, we systematically evaluate $k \in [2, 10]$ using the **Elbow Method (Inertia curve)** and quantitative evaluation metrics (**Silhouette Score**, **Davies-Bouldin Index**, and **Calinski-Harabasz Index**).

In [6]:
# Cell 5.1 — Fit initial K-Means (k=3)
K_INIT = 3

kmeans = KMeans(
    n_clusters=K_INIT,
    init='k-means++',
    n_init=10,
    max_iter=300,
    random_state=RANDOM_STATE,
)
kmeans.fit(X_scaled)
cluster_labels_k3 = kmeans.labels_

label_counts = dict(zip(*np.unique(cluster_labels_k3, return_counts=True)))
print(f"K-Means successfully fitted with k={K_INIT}")
print(f"Cluster label distribution: {label_counts}")
print(f"Inertia (Sum of squared distances): {kmeans.inertia_:.2f}")

K-Means successfully fitted with k=3


Cluster label distribution: {np.int32(0): np.int64(3927), np.int32(1): np.int64(2070), np.int32(2): np.int64(4003)}
Inertia (Sum of squared distances): 38298.31


## 6. Initial Validation & Cluster Profiling

In [7]:
# Cell 6.1 — Run validation checks using clustering_utils
validation_results = validate_clustering(
    cluster_labels=cluster_labels_k3,
    expected_k=K_INIT,
    feature_names=CLUSTER_FEATURES,
    n_samples=len(X_scaled),
)

print("=== Day 1 Validation Checks ===")
for check, status in validation_results.items():
    print(f"[{status}] {check}")

assert all(s == 'PASS' for s in validation_results.values()), "Validation checks failed!"

=== Day 1 Validation Checks ===
[PASS] Number of clusters equals k
[PASS] All clusters have >= 2 samples
[PASS] No cluster exceeds 95% of data
[PASS] Label leakage check (Ground truth labels excluded)
[PASS] Valid cluster assignment array


In [8]:
# Cell 6.2 — Compute cluster profile (mean feature values on original physical scale)
profile_df = profile_clusters(
    df_features=X_cluster,
    cluster_labels=cluster_labels_k3,
    true_labels=TRUE_LABELS,
)

print("=== Cluster Means (Original Physical Scale) ===")
display(profile_df) if 'display' in locals() or 'display' in globals() else print(profile_df)

=== Cluster Means (Original Physical Scale) ===
         Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
Cluster                                                                         
0                     301.73                   311.25                 1470.46   
1                     300.09                   310.08                 1795.99   
2                     298.27                   308.74                 1472.79   

         Torque [Nm]  Tool wear [min]  Type_encoded  Failure_Rate_Actual  
Cluster                                                                   
0              43.60           110.11          0.48                 0.05  
1              26.62           109.56          0.49                 0.02  
2              43.35           105.00          0.53                 0.02  


### Cluster Profile Observation
> **[STUDENT TASK — Day 1]:**  
> Look at the cluster means table above. Do the clusters separate by  
> temperature ranges, tool wear, or rotational speed? Does one cluster appear  
> to match high-wear, high-torque, or elevated thermal operating conditions?  
> Write 3–4 sentences describing what you actually observe in the physical sensor values.  
> Replace this prompt with your observation after reviewing the executed outputs.

---
## Day 1 Complete — Handoff Notes

**Milestone Summary:**
- Feature matrix constructed strictly from sensor and machine properties ($10000 \times 6$).
- Zero data leakage: Ground-truth failure target and subtypes excluded from clustering.
- StandardScaler applied.
- K-Means ($k=3$) fitted with reproducible seed ($42$); inertia $= 38298.31$.
- All sanity guard checks PASSED.

**Roadmap for Subsequent Milestones:**
- **Day 2:** Implement Elbow Method ($k=2$ to $k=10$), compute quantitative metrics (**Silhouette Score**, **Davies-Bouldin Index**, **Calinski-Harabasz Index**) across multiple $k$, and select optimal $k$.
- **Day 3:** Implement Agglomerative Hierarchical Clustering, plot Dendrogram, compare linkage methods (Ward, Complete, Average), and generate 2D PCA cluster visualizations and t-SNE plot.